In [ ]:
# -*- coding: utf-8 -*-


proyectoAA2

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/17ms1ezxUPy9binJH-qt9GrLqJgNsZo4Q

Cargamos google drive para acceder a los archivos del proyecto, esto permite leer el dataset guardado en drive


In [ ]:
import google as google
from google.colab import drive
drive.mount('/content/drive')


importamos librerias


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


cargamos el dataset entregado por el docente, este archivo contiene registros horarios de calidad del aire


In [ ]:
df=pd.read_csv('/content/drive/MyDrive/datasets/air_quality_docente.csv')


muestra de las primeras filas del dataset para verificar la carga


In [ ]:
df.head()

# Creamos una columna temporal completa combinando la fecha y la hora
# En este dataset, la columna "fecha" contiene el día y la columna "hora" contiene la hora del registro

df["fecha_hora"] = pd.to_datetime(df["fecha"] + " " + df["hora"])

# Mostramos las primeras filas para verificar que la columna "fecha_hora" fue creada correctamente
df.head()

# Ordenamos el DataFrame cronológicamente usando la columna "fecha_hora"
# Esto asegura que las observaciones queden desde la más antigua hasta la más reciente
df = df.sort_values("fecha_hora")

# Mostramos las primeras filas luego del ordenamiento
df.head()

# Verificamos si la columna "fecha_hora" está ordenada de menor a mayor
# El resultado debería ser True si el orden temporal es correcto
df["fecha_hora"].is_monotonic_increasing

# Definimos la columna "fecha_hora" como índice temporal del df, esto permite trabajar con operaciones propias de series temporales
# Con drop=True, la columna "fecha_hora" deja de aparecer como columna común y queda solo como índice
df = df.set_index("fecha_hora", drop=True).sort_index()

# Mostramos las primeras filas para confirmar que "fecha_hora" quedó como índice temporal
df.head()


se crea una copia del dataset original sin modifcar directamente el df


In [ ]:
df_tratado=df.copy()


muestra de las primeras filas


In [ ]:
df_tratado.head()

# Definimos el tamaño del conjunto de prueba
# En este caso reservamos el 20% final de la serie

n_test = int(len(df_tratado) * 0.2)

# Partición temporal
# Train queda con el 80% inicial y test con el 20% final

train = df_tratado.iloc[:-n_test].copy()
test = df_tratado.iloc[-n_test:].copy()

# Verificamos los períodos obtenidos

print("Conjunto de entrenamiento:")
print(train.index.min(), "→", train.index.max())
print("Cantidad de registros:", len(train))

print("\nConjunto de prueba:")
print(test.index.min(), "→", test.index.max())
print("Cantidad de registros:", len(test))

# Vista diaria del conjunto de entrenamiento
train_diario_vista = (train.assign(dia=train.index.floor("D"))
    .groupby("dia")["concentracion_benceno"]
    .mean()
    .reset_index())

# Vista diaria del conjunto de prueba
test_diario_vista = (test.assign(dia=test.index.floor("D"))
    .groupby("dia")["concentracion_benceno"]
    .mean()
    .reset_index())

plt.figure(figsize=(14, 5))

plt.plot(
    train_diario_vista["dia"],
    train_diario_vista["concentracion_benceno"],
    label="Entrenamiento")

plt.plot(
    test_diario_vista["dia"],
    test_diario_vista["concentracion_benceno"],
    label="Prueba")

plt.axvline(
    test_diario_vista["dia"].min(),
    linestyle="--",
    linewidth=1,
    label="Inicio del conjunto de prueba")

plt.title("Partición temporal de la concentración diaria de benceno")
plt.xlabel("Fecha")
plt.ylabel("Concentración diaria de benceno")
plt.legend()
plt.grid(True)
plt.show()

# Aplicamos resampling diario definiendo una forma de agregación para cada variable
# Como trabajamos con calidad del aire, usamos promedio diario para concentraciones, señales de sensores y variables ambientales

train_diario = train.resample("D").agg({
    "sensor_co": "mean",
    "concentracion_benceno": "mean",
    "sensor_nmhc": "mean",
    "sensor_nox": "mean",
    "sensor_no2": "mean",
    "sensor_o3": "mean",
    "temperatura": "mean",
    "humedad_relativa": "mean",
    "humedad_absoluta": "mean"
})

# Observamos el resultado
train_diario.head()


se selecciona la variable objetivo del proyecto que es concentracion de benceno desde el conjunto de entrenamiento dirario

El resultado se guarda en y_train, que será la serie temporal usada para entrenar el modelo univariante. Se usa .copy() para crear una copia independiente y evitar modificar accidentalmente los datos originales de train_diario.

Finalmente, y_train.head() muestra las primeras observaciones de la serie objetivo para verificar que se creó correctamente.


In [ ]:
y_train = train_diario["concentracion_benceno"].copy()

y_train.head()


Primero imprime el rango temporal de train_diario, mostrando la primera y la última fecha disponibles. Luego muestra la cantidad total de días en el conjunto de entrenamiento diario. Finalmente, revisa si existen valores faltantes en cada columna usando train_diario.isna().sum().

El resultado indica que train_diario va desde 2004-03-10 hasta 2005-01-16, contiene 313 días, y no tiene valores faltantes en ninguna variable. Esto significa que, después de agrupar los datos horarios a frecuencia diaria, el conjunto de entrenamiento quedó completo a nivel de valores.


In [ ]:
print("Train diario:")
print(train_diario.index.min(), "→", train_diario.index.max())
print("Cantidad de días:", len(train_diario))
print("Faltantes:")
print(train_diario.isna().sum())


Aunque train_diario no tenga valores faltantes, eso no garantiza que todos los promedios diarios hayan sido calculados con 24 horas completas. Por eso se cuenta cuántas observaciones horarias de concentracion_benceno hay por día.


In [ ]:
# Verificación de cantidad de registros horarios por día

conteo_horas_train = train["concentracion_benceno"].resample("D").count()

print("Primeros días:")
print(conteo_horas_train.head())

print("\nÚltimos días:")
print(conteo_horas_train.tail())

print("\nDías con menos de 24 registros horarios:")
print(conteo_horas_train[conteo_horas_train < 24])


El resultado muestra que la mayoría de los días tienen 24 registros, pero algunos días tienen menos

Esto significa que esos días están incompletos. El primer y el último día se explican por el inicio y el corte del conjunto de entrenamiento, pero también hay días incompletos en el medio de la serie. Para modelar de forma más consistente, conviene decidir si se eliminan esos días o si se aceptan los promedios calculados con menos horas. En este proyecto, la opción más prolija es conservar solo los días completos de 24 registros.


In [ ]:
# Conservamos solamente los días que tienen 24 registros horarios
# Esos son los días completos del conjunto de entrenamiento

dias_completos_train = conteo_horas_train[conteo_horas_train == 24].index

train_diario = train_diario.loc[dias_completos_train].copy()

print("Train diario luego de conservar solo días completos:")
print(train_diario.index.min(), "→", train_diario.index.max())
print("Cantidad de días:", len(train_diario))

train_diario.head()


Esta celda filtra el conjunto de entrenamiento diario para conservar únicamente los días que tienen 24 registros horarios. Como la serie original es horaria, un día completo debería tener 24 observaciones. Los días con menos registros pueden generar promedios diarios poco representativos, por eso se eliminan del conjunto de entrenamiento antes de continuar con el análisis y el modelado.


In [ ]:
# Actualizamos la variable objetivo luego de conservar solo días completos
# y_train será la serie diaria de concentración de benceno usada para análisis y modelado

y_train = train_diario["concentracion_benceno"].copy()

print("Cantidad de observaciones en y_train:", len(y_train))
print("Inicio:", y_train.index.min())
print("Fin:", y_train.index.max())

y_train.head()


Se asegura que la serie objetivo tenga dias completos, es decir, con 24 registros horarios, a partir de ahora, y_train representa la concentración diaria promedio de benceno que se usará para el análisis exploratorio y para entrenar el modelo univariante.


In [ ]:
# Graficamos la serie objetivo de entrenamiento
# Esto permite observar la evolución diaria de la concentración promedio de benceno

plt.figure(figsize=(14, 5))

plt.plot(y_train.index, y_train)

plt.title("Concentración diaria promedio de benceno - entrenamiento")
plt.xlabel("Fecha")
plt.ylabel("Concentración diaria promedio de benceno")
plt.grid(True)
plt.show()

# Calculamos una media móvil de 7 días sobre la serie objetivo
# Esto suaviza variaciones puntuales y permite observar mejor la tendencia general

media_movil_7 = y_train.rolling(window=7).mean()

plt.figure(figsize=(14, 5))

plt.plot(y_train.index, y_train, label="Serie diaria")
plt.plot(media_movil_7.index, media_movil_7, label="Media móvil 7 días")

plt.title("Concentración diaria de benceno con media móvil de 7 días")
plt.xlabel("Fecha")
plt.ylabel("Concentración diaria promedio de benceno")
plt.legend()
plt.grid(True)
plt.show()


Se calcula una media móvil de 7 días sobre la concentración diaria promedio de benceno. La media móvil suaviza las variaciones diarias y permite observar mejor la evolución general de la serie. En este caso, una ventana de 7 días es razonable porque el horizonte del proyecto es semanal y ayuda a analizar cambios en períodos cortos sin perder completamente la variabilidad de la serie.


In [ ]:
# Calculamos estadísticas descriptivas de la serie objetivo, esto permite conocer el nivel promedio, la variabilidad y los outliers

y_train.describe()

# Graficamos un boxplot
# Esto ayuda a detectar posibles valores atípicos en la concentración diaria de benceno

plt.figure(figsize=(8, 5))

plt.boxplot(y_train.dropna(), vert=True)

plt.title("Boxplot de la concentración diaria promedio de benceno")
plt.ylabel("Concentración diaria promedio de benceno")
plt.grid(True)
plt.show()

# Calculamos los límites del criterio IQR para detectar posibles valores atípicos

Q1 = y_train.quantile(0.25)
Q3 = y_train.quantile(0.75)
IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Límite inferior:", limite_inferior)
print("Límite superior:", limite_superior)

# Filtramos los posibles valores atípicos.

posibles_atipicos = y_train[
    (y_train < limite_inferior) | (y_train > limite_superior)
]

posibles_atipicos


Estos valores superan el límite superior calculado a partir del rango intercuartílico. No se detectaron valores atípicos bajos, ya que el límite inferior resultó negativo.

Estos valores no se eliminan automáticamente, porque en problemas de calidad del aire pueden representar episodios reales de mayor contaminación. Por lo tanto, se mantienen en la serie para conservar el comportamiento original de los datos.


In [ ]:
# Graficamos la serie de entrenamiento junto con los posibles valores atípicos, asi observamos en qué momentos aparecen los valores más altos

plt.figure(figsize=(14, 5))

plt.plot(y_train.index, y_train, label="Serie diaria")
plt.scatter(
    posibles_atipicos.index,
    posibles_atipicos,
    label="Posibles atípicos",
    zorder=3
)

plt.axhline(
    limite_superior,
    linestyle="--",
    linewidth=1,
    label="Límite superior IQR"
)

plt.title("Posibles valores atípicos en la concentración diaria de benceno")
plt.xlabel("Fecha")
plt.ylabel("Concentración diaria promedio de benceno")
plt.legend()
plt.grid(True)
plt.show()


revisamos patrones por dia de la semana, esto ayuda a ver si la concentracion cambia segun el calendario


In [ ]:
promedio_dia_semana = y_train.groupby(y_train.index.day_name()).mean()

orden_dias = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday"
]

promedio_dia_semana = promedio_dia_semana.reindex(orden_dias)

promedio_dia_semana


La concentración promedio de benceno es más alta entre martes, miércoles y jueves, con valores alrededor de 12, y baja bastante durante el fin de semana, especialmente el domingo, con un promedio cercano a 6.4.

Al agrupar la concentración diaria promedio de benceno por día de la semana, se observa un posible patrón semanal. Los valores medios son más altos entre martes y jueves, mientras que disminuyen durante el fin de semana, especialmente el domingo. Esto puede sugerir una relación entre la concentración de benceno y la actividad urbana o vehicular durante los días laborales.


In [ ]:
# Graficamos la concentración promedio de benceno por día de la semana, permite visualizar mejor el posible patrón semanal

plt.figure(figsize=(10, 5))

plt.bar(promedio_dia_semana.index,promedio_dia_semana.values)

plt.title("Concentración promedio de benceno por día de la semana")
plt.xlabel("Día de la semana")
plt.ylabel("Concentración diaria promedio de benceno")
plt.xticks(rotation=45)
plt.grid(axis="y")
plt.show()

# Analizamos la concentración promedio de benceno por mes, esto nos permite observar si existen cambios asociados al período del año

promedio_mes = y_train.groupby(y_train.index.month).mean()

promedio_mes


no aparece el mes 2 porque tu conjunto de entrenamiento termina en enero de 2005, y no llega a febrero. También el mes 1 corresponde solo a parte de enero de 2005.

Al agrupar la concentración diaria promedio de benceno por mes, se observan diferencias entre períodos del año. El mes con menor promedio es agosto, mientras que los valores más altos aparecen en octubre y noviembre. Esto sugiere que la serie puede presentar variaciones asociadas al período del año. Sin embargo, la comparación debe interpretarse con cuidado porque no todos los meses tienen la misma cantidad de días en el conjunto de entrenamiento.


In [ ]:
plt.figure(figsize=(10, 5))

plt.bar(
    promedio_mes.index,
    promedio_mes.values
)

plt.title("Concentración promedio de benceno por mes")
plt.xlabel("Mes")
plt.ylabel("Concentración diaria promedio de benceno")
plt.xticks(promedio_mes.index)
plt.grid(axis="y")
plt.show()

# importamos la función para graficar la autocorrelación la ACF permite observar la relación de la serie con sus propios valores pasados

from statsmodels.graphics.tsaplots import plot_acf

plt.figure(figsize=(10, 5))

plot_acf(y_train,lags=30)

plt.title("ACF - Concentración diaria promedio de benceno")
plt.xlabel("Rezagos")
plt.ylabel("Autocorrelación")
plt.grid(True)
plt.show()


El lag 1 es alto, cerca de 0.6 eso quiere decir que la concentración de benceno de un día se parece bastante a la del día anterior. En términos simples: si ayer hubo concentración alta, hoy tiende a seguir relativamente alta; si ayer fue baja, hoy tiende a seguir baja. Después aparecen picos en lag 7, 14, 21 y 28. Eso es importante porque sugiere un posible patrón semanal. Como la serie es diaria, cada 7 rezagos representa una semana. Entonces la concentración de hoy puede parecerse a la de hace una semana, dos semanas, tres semanas, etc.

La zona azul clara es una banda de confianza. Cuando una barra sobresale bastante de esa zona, la autocorrelación de ese rezago parece más relevante. Si queda dentro de la zona, puede ser ruido.


In [ ]:
from statsmodels.graphics.tsaplots import plot_pacf

plt.figure(figsize=(10, 5))

plot_pacf(y_train,lags=30,method="ywm")

plt.title("PACF - Concentración diaria promedio de benceno")
plt.xlabel("Rezagos")
plt.ylabel("Autocorrelación parcial")
plt.grid(True)
plt.show()


La PACF muestra una autocorrelación parcial fuerte en el rezago 1, lo que indica que la concentración del día anterior tiene una relación directa importante con la concentración actual. También aparecen algunos rezagos relevantes en torno a períodos cercanos a una semana, como 6, 13, 20 y 27 días. Esto complementa lo observado en la ACF y sugiere que el modelo univariante debería considerar lags recientes y lags relacionados con ciclos semanales.


In [ ]:
!pip install skforecast -q

from skforecast.recursive import ForecasterRecursive
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Verificamos si la serie objetivo tiene una frecuencia diaria regular.
# Esto es importante porque el modelo de forecasting trabaja con pasos temporales ordenados y consistentes.

print("Frecuencia inferida:", y_train.index.inferred_freq)

indice_diario_esperado = pd.date_range(
    start=y_train.index.min(),
    end=y_train.index.max(),
    freq="D"
)

fechas_faltantes = indice_diario_esperado.difference(y_train.index)

print("Cantidad de fechas esperadas:", len(indice_diario_esperado))
print("Cantidad de fechas reales:", len(y_train))
print("Cantidad de fechas faltantes:", len(fechas_faltantes))

fechas_faltantes


Al realizar el resampling diario, primero se revisó cuántos registros horarios tenía cada día del conjunto de entrenamiento. Esta verificación se hizo porque la serie original tiene frecuencia horaria y, al pasarla a frecuencia diaria, un día completo debería estar construido a partir de 24 observaciones.

El objetivo de esta revisión fue evaluar la calidad de los promedios diarios generados. Aunque el resampling no produjo valores faltantes, se detectaron algunos días con menos de 24 registros horarios. Esto significa que esos promedios diarios fueron calculados con menos información que un día completo.

Una alternativa posible era eliminar los días incompletos y conservar únicamente los días con 24 registros horarios. Sin embargo, se rechazó esta opción porque al eliminar días intermedios se generan huecos en la serie diaria. Esto rompe la frecuencia temporal regular, ya que la serie deja de tener una observación por cada día calendario.

Para un problema de forecasting, mantener una frecuencia diaria regular es importante porque cada paso temporal debe representar siempre la misma unidad de tiempo: un día. Por esta razón, se decidió mantener los días incompletos y calcular el promedio diario con las observaciones disponibles.

Los días incompletos se documentan como una limitación del dataset, pero no se eliminan para preservar la continuidad temporal necesaria para el modelado.


In [ ]:
# Reconstruimos el conjunto diario de entrenamiento sin eliminar días
# Aunque algunos días tienen menos de 24 registros horarios, se mantienen para conservar una frecuencia diaria regular
# Usamos promedio diario porque las variables representan concentraciones, sensores y condiciones ambientales

train_diario = train.resample("D").agg({"sensor_co": "mean","concentracion_benceno": "mean","sensor_nmhc": "mean","sensor_nox": "mean",
    "sensor_no2": "mean","sensor_o3": "mean","temperatura": "mean","humedad_relativa": "mean","humedad_absoluta": "mean"})

y_train = train_diario["concentracion_benceno"].copy()

print("Frecuencia inferida:", y_train.index.inferred_freq)
print("Cantidad de días:", len(y_train))
print("Faltantes en y_train:", y_train.isna().sum())

train_diario.head()


Esta celda reconstruye el conjunto de entrenamiento diario manteniendo todos los días del calendario. Aunque algunos días tienen menos de 24 registros horarios, se conservan para no romper la frecuencia diaria regular. La agregación se realiza mediante promedio diario porque las variables del dataset representan concentraciones, señales de sensores y condiciones ambientales, no conteos acumulativos.

La variable y_train se vuelve a definir a partir de train_diario, usando la columna concentracion_benceno, que es la variable objetivo del proyecto.


In [ ]:
# Verificamos la serie objetivo reconstruida.
# Esta serie mantiene todos los días del calendario y será usada para análisis y modelado.

print("Inicio:", y_train.index.min())
print("Fin:", y_train.index.max())
print("Cantidad de observaciones:", len(y_train))
print("Frecuencia inferida:", y_train.index.inferred_freq)
print("Valores faltantes:", y_train.isna().sum())

y_train.head()

# Creamos un modelo base univariante con skforecast.
# Este modelo usa solamente la historia pasada de la variable objetivo.
# Usamos 7 lags porque trabajamos con frecuencia diaria y el horizonte del proyecto es semanal.

forecaster_base = ForecasterRecursive(
    RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ),
    lags=7
)

# Entrenamos el modelo usando únicamente y_train.
forecaster_base.fit(y=y_train)

forecaster_base


Esta celda construye el primer modelo base univariante. Se usa ForecasterRecursive, que transforma la serie temporal en un problema supervisado utilizando valores pasados como variables predictoras. El modelo usa solamente la historia de concentracion_benceno, sin incorporar todavía variables exógenas.

Se define lags=7 porque la serie está en frecuencia diaria y se quiere que el modelo tenga disponible la información de la última semana para predecir valores futuros. Como regresor base se utiliza RandomForestRegressor con una configuración inicial simple, sin ajuste de hiperparámetros.


In [ ]:
# Generamos las primeras predicciones del modelo base.
# Como el horizonte del proyecto es de 7 días, pedimos 7 pasos futuros.

predicciones_base = forecaster_base.predict(steps=7)

predicciones_base

# Aplicamos el mismo resampling diario al conjunto de prueba.
# No usamos test para ajustar el modelo, solo para evaluar la predicción final.

test_diario = test.resample("D").agg({
    "sensor_co": "mean",
    "concentracion_benceno": "mean",
    "sensor_nmhc": "mean",
    "sensor_nox": "mean",
    "sensor_no2": "mean",
    "sensor_o3": "mean",
    "temperatura": "mean",
    "humedad_relativa": "mean",
    "humedad_absoluta": "mean"
})

y_test = test_diario["concentracion_benceno"].copy()

print("Inicio test:", y_test.index.min())
print("Fin test:", y_test.index.max())
print("Cantidad de días test:", len(y_test))
print("Faltantes en y_test:", y_test.isna().sum())


Esta celda transforma el conjunto de prueba a frecuencia diaria usando el mismo criterio aplicado al conjunto de entrenamiento. Como las variables representan concentraciones, sensores y condiciones ambientales, se utiliza el promedio diario. Luego se selecciona concentracion_benceno como variable objetivo de prueba, guardándola en y_test.

El conjunto de prueba no se usa para entrenar ni ajustar el modelo. Se reserva únicamente para comparar las predicciones generadas por el modelo contra valores reales no usados durante el entrenamiento.


In [ ]:
# Seleccionamos los primeros 7 días reales del conjunto de prueba.
# Estos valores se comparan contra las 7 predicciones generadas por el modelo base.

y_test_7 = y_test.iloc[:7]

# Calculamos métricas de error.
# MAE mide el error absoluto promedio.
# RMSE penaliza más los errores grandes.

mae_base = mean_absolute_error(y_test_7, predicciones_base)
rmse_base = np.sqrt(mean_squared_error(y_test_7, predicciones_base))

print("MAE modelo base:", mae_base)
print("RMSE modelo base:", rmse_base)


El modelo base univariante obtuvo un MAE de 4.63 y un RMSE de 5.31 al predecir los primeros 7 días del conjunto de prueba. Esto significa que, en promedio, las predicciones se alejaron aproximadamente 4.63 unidades de concentración de benceno respecto a los valores reales. El RMSE resultó mayor que el MAE, lo que indica que algunos errores fueron más grandes y tuvieron mayor peso en la métrica cuadrática.

Esta evaluación inicial permite comprobar que el flujo de modelado funciona, pero todavía corresponde tomarla como una evaluación puntual. Para obtener una evaluación más robusta será necesario aplicar backtesting temporal.


In [ ]:
# Graficamos la predicción inicial del modelo base frente a los valores reales.
# También mostramos los últimos días del entrenamiento para dar contexto temporal.

plt.figure(figsize=(12, 5))

plt.plot(
    y_train.index[-30:],
    y_train.iloc[-30:],
    label="Últimos 30 días de entrenamiento"
)

plt.plot(
    y_test_7.index,
    y_test_7,
    label="Valores reales test"
)

plt.plot(
    y_test_7.index,
    predicciones_base,
    label="Predicción modelo base"
)

plt.title("Predicción inicial del modelo base univariante")
plt.xlabel("Fecha")
plt.ylabel("Concentración diaria promedio de benceno")
plt.legend()
plt.grid(True)
plt.show()


Este gráfico compara las predicciones iniciales del modelo base univariante con los valores reales de los primeros 7 días del conjunto de prueba. También se muestran los últimos 30 días del conjunto de entrenamiento para observar el contexto temporal previo a la predicción.

La visualización permite evaluar si el modelo sigue la dirección general de la serie y si las predicciones se acercan razonablemente a los valores observados. Esta comparación es una primera evaluación puntual; más adelante se utilizará backtesting para obtener una evaluación temporal más robusta.


In [ ]:
# Importamos las herramientas de backtesting de skforecast.
# TimeSeriesFold define cómo se hacen los cortes temporales.
# backtesting_forecaster evalúa el modelo siguiendo esos cortes.

from skforecast.model_selection import TimeSeriesFold, backtesting_forecaster

# Definimos el protocolo de backtesting interno.
# steps=7 porque el horizonte del proyecto es de 7 días.
# initial_train_size usa el 70% inicial de y_train para el primer entrenamiento.
# refit=True indica que el modelo se reentrena en cada corte temporal.
# fixed_train_size=False indica ventana expansiva: el entrenamiento crece con el tiempo.

initial_train_size = int(len(y_train) * 0.7)

cv = TimeSeriesFold(
    steps=7,
    initial_train_size=initial_train_size,
    refit=True,
    fixed_train_size=False,
    gap=0,
    allow_incomplete_fold=True
)

# Evaluamos el modelo base univariante mediante backtesting interno.
# Esta evaluación se realiza solo dentro de y_train, sin usar el conjunto de test.

metricas_backtesting_base, predicciones_backtesting_base = backtesting_forecaster(
    forecaster=forecaster_base,
    y=y_train,
    cv=cv,
    metric=["mean_absolute_error", "mean_squared_error"],
    n_jobs="auto",
    verbose=False,
    show_progress=True
)

metricas_backtesting_base

# Convertimos el error cuadrático medio a RMSE.
# El RMSE queda en la misma escala que la concentración de benceno.

metricas_backtesting_base = metricas_backtesting_base.copy()

metricas_backtesting_base["root_mean_squared_error"] = np.sqrt(
    metricas_backtesting_base["mean_squared_error"]
)

metricas_backtesting_base

# Graficamos las predicciones obtenidas mediante backtesting.
# Esto permite comparar visualmente los valores reales de entrenamiento con las predicciones del modelo base.

plt.figure(figsize=(14, 5))

plt.plot(
    y_train.index,
    y_train,
    label="Valores reales de entrenamiento"
)

plt.plot(
    predicciones_backtesting_base.index,
    predicciones_backtesting_base["pred"],
    label="Predicciones backtesting"
)

plt.title("Backtesting del modelo base univariante")
plt.xlabel("Fecha")
plt.ylabel("Concentración diaria promedio de benceno")
plt.legend()
plt.grid(True)
plt.show()


El modelo base univariante obtuvo en backtesting interno un MAE de 4.61 y un RMSE de 5.95. Esto significa que, en promedio, las predicciones se alejaron aproximadamente 4.61 unidades de concentración de benceno respecto a los valores reales del conjunto de entrenamiento validado temporalmente.

El RMSE es mayor que el MAE, lo que indica que algunos errores fueron relativamente grandes. Esta evaluación es más confiable que la predicción puntual inicial, porque el modelo fue evaluado en varios cortes temporales dentro del conjunto de entrenamiento.


In [ ]:
# Definimos una grilla pequeña y razonable de configuraciones.
# Probamos distintas ventanas de lags y algunos hiperparámetros del Random Forest.

resultados_grid = []

lags_grid = [7, 14, 21, 28]

param_grid = [
    {"n_estimators": 100, "max_depth": 3, "min_samples_leaf": 1},
    {"n_estimators": 100, "max_depth": 5, "min_samples_leaf": 1},
    {"n_estimators": 100, "max_depth": None, "min_samples_leaf": 1},
    {"n_estimators": 100, "max_depth": None, "min_samples_leaf": 3}
]

for lags in lags_grid:
    for params in param_grid:

        forecaster = ForecasterRecursive(
            RandomForestRegressor(
                n_estimators=params["n_estimators"],
                max_depth=params["max_depth"],
                min_samples_leaf=params["min_samples_leaf"],
                random_state=42
            ),
            lags=lags
        )

        metricas, predicciones = backtesting_forecaster(
            forecaster=forecaster,
            y=y_train,
            cv=cv,
            metric=["mean_absolute_error", "mean_squared_error"],
            n_jobs="auto",
            verbose=False,
            show_progress=False
        )

        mae = metricas.loc[0, "mean_absolute_error"]
        mse = metricas.loc[0, "mean_squared_error"]
        rmse = np.sqrt(mse)

        resultados_grid.append({
            "lags": lags,
            "n_estimators": params["n_estimators"],
            "max_depth": params["max_depth"],
            "min_samples_leaf": params["min_samples_leaf"],
            "mae": mae,
            "rmse": rmse
        })

resultados_grid = pd.DataFrame(resultados_grid)

resultados_grid.sort_values("mae")


El Grid Search temporal evaluó distintas configuraciones de lags e hiperparámetros del Random Forest usando el mismo protocolo de backtesting interno. La mejor configuración según MAE fue la que utilizó 28 lags, 100 árboles, profundidad máxima sin limitar y min_samples_leaf = 3.

Esta configuración obtuvo un MAE de 4.36 y un RMSE de 5.58, mejorando el modelo base, que había obtenido un MAE de 4.61 y un RMSE de 5.95. La elección de 28 lags es coherente con lo observado en la ACF, donde aparecían autocorrelaciones relevantes en rezagos semanales como 7, 14, 21 y 28 días.


In [ ]:
# Seleccionamos la mejor configuración encontrada en el Grid Search.
# Esta configuración queda fija para la evaluación final.

mejor_forecaster = ForecasterRecursive(
    RandomForestRegressor(
        n_estimators=100,
        max_depth=None,
        min_samples_leaf=3,
        random_state=42
    ),
    lags=28
)

# Entrenamos el modelo ajustado con todo y_train.
mejor_forecaster.fit(y=y_train)

mejor_forecaster


Esta celda construye el modelo univariante ajustado usando la mejor configuración encontrada en el Grid Search temporal. A partir de este momento, los lags y los hiperparámetros quedan fijos. El modelo se entrena con todo el conjunto de entrenamiento diario (`y_train`) y queda preparado para la evaluación final sobre el conjunto de prueba.


In [ ]:
# Definimos la mejor configuración encontrada en el Grid Search

best_lags = 28

best_params = {
    "n_estimators": 100,
    "max_depth": None,
    "min_samples_leaf": 3,
    "random_state": 42
}

best_params

# Revisamos valores nulos en train y test diarios

print("Nulos en y_train:", y_train.isna().sum())
print("Nulos en y_test:", y_test.isna().sum())

# Creamos copias específicas para el modelo final

y_train_best = y_train.copy()
y_test_best = y_test.copy()

# Tratamiento de nulos solo si existen días completos sin datos

if y_train_best.isna().sum() > 0:
    y_train_best = y_train_best.interpolate(method="time").ffill().bfill()

if y_test_best.isna().sum() > 0:
    y_test_best = y_test_best.interpolate(method="time").ffill().bfill()

print("Nulos finales en y_train_best:", y_train_best.isna().sum())
print("Nulos finales en y_test_best:", y_test_best.isna().sum())

# Creamos el mejor modelo univariante

forecaster_best = ForecasterRecursive(
    RandomForestRegressor(**best_params),
    lags=best_lags
)

# Entrenamos usando todo el conjunto de entrenamiento diario

forecaster_best.fit(y=y_train_best)

forecaster_best

# Predicción de los próximos 7 días

steps = 7

predicciones_best = forecaster_best.predict(steps=steps)

predicciones_best

# Alineamos valores reales y predichos por fecha

evaluacion_best = pd.concat(
    [
        y_test_best.reindex(predicciones_best.index).rename("real"),
        predicciones_best.rename("prediccion")
    ],
    axis=1
)

evaluacion_best

# Métricas del mejor modelo sobre test

mae_best_test = mean_absolute_error(
    evaluacion_best["real"],
    evaluacion_best["prediccion"]
)

mse_best_test = mean_squared_error(
    evaluacion_best["real"],
    evaluacion_best["prediccion"]
)

rmse_best_test = np.sqrt(mse_best_test)

metricas_best_test = pd.DataFrame({
    "modelo": ["Mejor modelo univariante"],
    "lags": [best_lags],
    "n_estimators": [best_params["n_estimators"]],
    "max_depth": [best_params["max_depth"]],
    "min_samples_leaf": [best_params["min_samples_leaf"]],
    "MAE": [mae_best_test],
    "MSE": [mse_best_test],
    "RMSE": [rmse_best_test]
})

metricas_best_test


Comparamos el desempeño del modelo base con el mejor modelo univariante ajustado. El modelo base utilizaba 7 lags, mientras que el modelo seleccionado por Grid Search utiliza 28 lags y una configuración más regularizada del Random Forest.


In [ ]:
# Gráfico de valores reales vs predichos en test

plt.figure(figsize=(10, 5))

plt.plot(
    evaluacion_best.index,
    evaluacion_best["real"],
    marker="o",
    label="Valor real"
)

plt.plot(
    evaluacion_best.index,
    evaluacion_best["prediccion"],
    marker="o",
    label="Predicción"
)

plt.title("Mejor modelo univariante: predicción de concentración de benceno")
plt.xlabel("Fecha")
plt.ylabel("Concentración de benceno")
plt.legend()
plt.grid(True)
plt.show()


Analizamos los errores de predicción para identificar si el modelo tiende a sobreestimar o subestimar la concentración de benceno.


In [ ]:
# Cálculo de errores

evaluacion_best["error"] = evaluacion_best["real"] - evaluacion_best["prediccion"]
evaluacion_best["error_absoluto"] = evaluacion_best["error"].abs()

evaluacion_best

# Gráfico de errores

plt.figure(figsize=(10, 4))

plt.bar(
    evaluacion_best.index,
    evaluacion_best["error"]
)

plt.axhline(
    y=0,
    linestyle="--"
)

plt.title("Errores de predicción del mejor modelo univariante")
plt.xlabel("Fecha")
plt.ylabel("Error real - predicción")
plt.grid(True)
plt.show()


## Continuacion del proyecto: modelo con variables exogenas, comparacion final y conclusiones

Hasta este punto ya se construyo y evaluo el mejor modelo univariante. A continuacion se incorpora un modelo con variables exogenas, como pide la ficha del grupo, para comparar si el uso de sensores y variables ambientales mejora la prediccion de la concentracion diaria de benceno.


In [ ]:
# Definimos las variables exogenas disponibles en el dataset.
# No incluimos la variable objetivo, porque esa es la serie que queremos predecir.

variables_exogenas = [
    "sensor_co",
    "sensor_nmhc",
    "sensor_nox",
    "sensor_no2",
    "sensor_o3",
    "temperatura",
    "humedad_relativa",
    "humedad_absoluta"
]

# Creamos los conjuntos de exogenas para entrenamiento y prueba.
# Usamos las mismas fechas que y_train_best e y_test_best para mantener alineacion temporal.

exog_train = train_diario[variables_exogenas].copy()
exog_test = test_diario[variables_exogenas].copy()

# Tratamiento preventivo de valores faltantes en variables exogenas.
# Se usa interpolacion temporal y luego ffill/bfill para asegurar que no queden nulos.

exog_train = exog_train.interpolate(method="time").ffill().bfill()
exog_test = exog_test.interpolate(method="time").ffill().bfill()

print("Nulos en exog_train:")
print(exog_train.isna().sum())

print("\nNulos en exog_test:")
print(exog_test.isna().sum())

exog_train.head()


Las variables exogenas seleccionadas corresponden a sensores de contaminantes y variables ambientales. Estas variables pueden aportar informacion adicional para explicar cambios en la concentracion de benceno. En la evaluacion sobre test se usan los valores observados de esas variables para los primeros 7 dias de prueba.


In [ ]:
# Definimos una grilla sencilla para el modelo con variables exogenas.
# Se mantiene Random Forest para comparar de forma justa con el modelo univariante.

resultados_grid_exog = []

lags_grid_exog = [7, 14, 28]

param_grid_exog = [
    {"n_estimators": 100, "max_depth": 5, "min_samples_leaf": 1},
    {"n_estimators": 100, "max_depth": None, "min_samples_leaf": 3}
]

for lags in lags_grid_exog:
    for params in param_grid_exog:

        forecaster_exog = ForecasterRecursive(
            RandomForestRegressor(
                n_estimators=params["n_estimators"],
                max_depth=params["max_depth"],
                min_samples_leaf=params["min_samples_leaf"],
                random_state=42
            ),
            lags=lags
        )

        metricas_exog, predicciones_exog_bt = backtesting_forecaster(
            forecaster=forecaster_exog,
            y=y_train_best,
            exog=exog_train,
            cv=cv,
            metric=["mean_absolute_error", "mean_squared_error"],
            n_jobs="auto",
            verbose=False,
            show_progress=False
        )

        mae = metricas_exog.loc[0, "mean_absolute_error"]
        mse = metricas_exog.loc[0, "mean_squared_error"]
        rmse = np.sqrt(mse)

        resultados_grid_exog.append({
            "lags": lags,
            "n_estimators": params["n_estimators"],
            "max_depth": params["max_depth"],
            "min_samples_leaf": params["min_samples_leaf"],
            "mae": mae,
            "rmse": rmse
        })

resultados_grid_exog = pd.DataFrame(resultados_grid_exog)

resultados_grid_exog.sort_values("mae")


El Grid Search del modelo con exogenas permite comparar distintas cantidades de rezagos y configuraciones del Random Forest. Esta busqueda mantiene una validacion temporal mediante backtesting, por lo que respeta el orden cronologico de la serie.


In [ ]:
# Seleccionamos la mejor configuracion del modelo con exogenas segun MAE.

mejor_exog = resultados_grid_exog.sort_values("mae").iloc[0]

best_lags_exog = int(mejor_exog["lags"])

best_params_exog = {
    "n_estimators": int(mejor_exog["n_estimators"]),
    "max_depth": None if pd.isna(mejor_exog["max_depth"]) else int(mejor_exog["max_depth"]),
    "min_samples_leaf": int(mejor_exog["min_samples_leaf"]),
    "random_state": 42
}

print("Mejor configuracion con exogenas:")
print("Lags:", best_lags_exog)
print("Parametros:", best_params_exog)

# Entrenamos el mejor modelo con exogenas usando todo el conjunto de entrenamiento.

forecaster_exog_best = ForecasterRecursive(
    RandomForestRegressor(**best_params_exog),
    lags=best_lags_exog
)

forecaster_exog_best.fit(
    y=y_train_best,
    exog=exog_train
)

forecaster_exog_best

# Prediccion de los primeros 7 dias del conjunto de prueba usando variables exogenas observadas.

exog_test_7 = exog_test.iloc[:steps].copy()

predicciones_exog_best = forecaster_exog_best.predict(
    steps=steps,
    exog=exog_test_7
)

# Alineamos el indice con las fechas reales de test.

predicciones_exog_best.index = y_test_7.index

predicciones_exog_best

# Evaluacion del modelo con exogenas sobre los primeros 7 dias de test.

mae_exog_test = mean_absolute_error(
    y_test_7,
    predicciones_exog_best
)

mse_exog_test = mean_squared_error(
    y_test_7,
    predicciones_exog_best
)

rmse_exog_test = np.sqrt(mse_exog_test)

metricas_exog_test = pd.DataFrame({
    "modelo": ["Modelo con variables exogenas"],
    "lags": [best_lags_exog],
    "n_estimators": [best_params_exog["n_estimators"]],
    "max_depth": [best_params_exog["max_depth"]],
    "min_samples_leaf": [best_params_exog["min_samples_leaf"]],
    "MAE": [mae_exog_test],
    "MSE": [mse_exog_test],
    "RMSE": [rmse_exog_test]
})

metricas_exog_test


Ahora se comparan los tres resultados principales: el modelo base univariante, el mejor modelo univariante ajustado y el modelo que incorpora variables exogenas.


In [ ]:
# Construimos una tabla comparativa final.

metricas_base_test = pd.DataFrame({
    "modelo": ["Modelo base univariante"],
    "lags": [7],
    "n_estimators": [100],
    "max_depth": [None],
    "min_samples_leaf": [None],
    "MAE": [mae_base],
    "MSE": [mean_squared_error(y_test_7, predicciones_base)],
    "RMSE": [rmse_base]
})

comparacion_modelos = pd.concat(
    [metricas_base_test, metricas_best_test, metricas_exog_test],
    ignore_index=True
).sort_values("MAE")

comparacion_modelos

# Creamos una tabla con valores reales y predicciones de cada modelo.

evaluacion_final = pd.DataFrame({
    "real": y_test_7,
    "prediccion_base_univariante": predicciones_base.values,
    "prediccion_mejor_univariante": evaluacion_best["prediccion"].values,
    "prediccion_exogenas": predicciones_exog_best.values
}, index=y_test_7.index)

evaluacion_final["error_base_univariante"] = evaluacion_final["real"] - evaluacion_final["prediccion_base_univariante"]
evaluacion_final["error_mejor_univariante"] = evaluacion_final["real"] - evaluacion_final["prediccion_mejor_univariante"]
evaluacion_final["error_exogenas"] = evaluacion_final["real"] - evaluacion_final["prediccion_exogenas"]

evaluacion_final

# Grafico comparativo de predicciones finales.

plt.figure(figsize=(12, 5))

plt.plot(
    evaluacion_final.index,
    evaluacion_final["real"],
    marker="o",
    label="Valor real"
)

plt.plot(
    evaluacion_final.index,
    evaluacion_final["prediccion_base_univariante"],
    marker="o",
    label="Modelo base univariante"
)

plt.plot(
    evaluacion_final.index,
    evaluacion_final["prediccion_mejor_univariante"],
    marker="o",
    label="Mejor modelo univariante"
)

plt.plot(
    evaluacion_final.index,
    evaluacion_final["prediccion_exogenas"],
    marker="o",
    label="Modelo con exogenas"
)

plt.title("Comparacion de predicciones sobre los primeros 7 dias de test")
plt.xlabel("Fecha")
plt.ylabel("Concentracion diaria promedio de benceno")
plt.legend()
plt.grid(True)
plt.show()

# Grafico de errores del modelo con exogenas.

plt.figure(figsize=(10, 4))

plt.bar(
    evaluacion_final.index,
    evaluacion_final["error_exogenas"]
)

plt.axhline(
    y=0,
    linestyle="--"
)

plt.title("Errores de prediccion del modelo con variables exogenas")
plt.xlabel("Fecha")
plt.ylabel("Error real - prediccion")
plt.grid(True)
plt.show()


El modelo con variables exogenas se evalua con los valores reales de sensores y variables ambientales disponibles en el conjunto de prueba. Esto permite medir si esas variables ayudan a mejorar la prediccion. Sin embargo, para pronosticar dias futuros reales con exogenas seria necesario conocer o estimar previamente esas variables futuras.


In [ ]:
# Guardamos resultados y modelos para que el proyecto sea reproducible.

import os
import joblib

os.makedirs("data/processed", exist_ok=True)
os.makedirs("models", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

# Dataset diario procesado completo.

df_diario_completo = df_tratado.resample("D").agg({
    "sensor_co": "mean",
    "concentracion_benceno": "mean",
    "sensor_nmhc": "mean",
    "sensor_nox": "mean",
    "sensor_no2": "mean",
    "sensor_o3": "mean",
    "temperatura": "mean",
    "humedad_relativa": "mean",
    "humedad_absoluta": "mean"
})

df_diario_completo = df_diario_completo.interpolate(method="time").ffill().bfill()

df_diario_completo.to_csv("data/processed/air_quality_diario_procesado.csv")
comparacion_modelos.to_csv("outputs/comparacion_modelos.csv", index=False)
evaluacion_final.to_csv("outputs/evaluacion_final_test_7_dias.csv")
resultados_grid.to_csv("outputs/grid_search_univariante.csv", index=False)
resultados_grid_exog.to_csv("outputs/grid_search_exogenas.csv", index=False)

# Guardamos los modelos entrenados.

joblib.dump(forecaster_best, "models/mejor_modelo_univariante.pkl")
joblib.dump(forecaster_exog_best, "models/mejor_modelo_exogenas.pkl")

print("Archivos guardados correctamente.")


## Pronostico final operativo para los proximos 7 dias

El modelo con exogenas obtuvo una comparacion util sobre test, pero para predecir fechas futuras reales necesita conocer valores futuros de sensores y variables ambientales. Por eso, para generar un pronostico operativo de proximos 7 dias se usa el mejor modelo univariante, entrenado nuevamente con toda la serie diaria disponible.


In [ ]:
# Entrenamos un modelo final univariante con toda la historia disponible.

y_completo = df_diario_completo["concentracion_benceno"].copy()

forecaster_final = ForecasterRecursive(
    RandomForestRegressor(**best_params),
    lags=best_lags
)

forecaster_final.fit(y=y_completo)

pronostico_7_dias = forecaster_final.predict(steps=7)
pronostico_7_dias.name = "pronostico_concentracion_benceno"

pronostico_7_dias.to_csv("outputs/pronostico_proximos_7_dias.csv")
joblib.dump(forecaster_final, "models/modelo_final_univariante.pkl")

pronostico_7_dias


## Conclusiones

El proyecto permitio construir un flujo completo de forecasting para la concentracion diaria promedio de benceno. Primero se preparo la serie temporal respetando el orden cronologico, se paso de frecuencia horaria a diaria y se separo el conjunto de entrenamiento y prueba sin mezclar el futuro con el pasado.

El modelo base univariante sirvio como punto de comparacion inicial. Luego, mediante backtesting y Grid Search, se selecciono un mejor modelo univariante con mas rezagos e hiperparametros ajustados. Finalmente, se incorporaron variables exogenas relacionadas con sensores de contaminantes y condiciones ambientales.

La comparacion final muestra que el modelo con variables exogenas puede mejorar la prediccion cuando se dispone de esas variables para el periodo a estimar. Como limitacion, para usar ese modelo en un pronostico futuro real seria necesario conocer o predecir previamente las variables exogenas futuras. Por ese motivo, tambien se guardo un modelo final univariante operativo para generar el pronostico de los proximos 7 dias usando unicamente la historia de la concentracion de benceno.


## Uso de IA generativa

Durante el desarrollo del trabajo se utilizo una herramienta de IA generativa como apoyo puntual para revisar la continuidad del notebook, realizar ajustes en el codigo, mejorar la organizacion de algunas secciones y redactar explicaciones finales. Las decisiones de preparacion de datos, modelado, evaluacion e interpretacion fueron revisadas por el grupo antes de incorporarse al proyecto.

| Prompt utilizado | Respuesta obtenida / uso en el proyecto |
|---|---|
| "Revisar la continuidad del analisis de series temporales y sugerir como cerrar la etapa final del notebook" | Se recibieron sugerencias para ordenar la parte final del flujo de trabajo, incluyendo comparacion de modelos, uso de variables exogenas y conclusiones. |
| "Ayudar a completar el codigo manteniendo las variables y el enfoque que ya estaban en el notebook" | Se obtuvo apoyo para agregar celdas de codigo compatibles con el desarrollo previo, especialmente para el modelo con variables exogenas, comparacion de metricas y guardado de resultados. |
| "Ayudar a redactar una explicacion breve para la comparacion entre el modelo univariante y el modelo con variables adicionales" | Se obtuvo una redaccion base que luego fue adaptada al contexto del dataset y a los resultados obtenidos. |
| "Sugerir una forma clara de presentar metricas, predicciones y conclusiones en el notebook" | Se usaron recomendaciones de estructura para presentar tablas de metricas, graficos comparativos y una conclusion final. |
| "Convertir el desarrollo a formato notebook manteniendo las secciones ya trabajadas" | Se genero apoyo para conservar el orden del trabajo y dejarlo en formato `.ipynb` ejecutable. |

El uso de IA fue de apoyo para ajustes de codigo, organizacion, redaccion y revision del cierre del notebook. El contenido fue revisado y ajustado para que mantuviera coherencia con el desarrollo realizado por el grupo.
